<table style="width: 100%; border-collapse: collapse; border: none; background: #fffbeb; border-left: 6px solid #f59e0b; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #78350f; font-size: 2em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        💡 Contar los Votos en Varias Mesas: la Idea de MapReduce 🗳️
      </h1>
      <p style="margin: 6px 0 0 0; color: #b45309; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Big Data
      </p>
      <p style="margin: 4px 0 0 0; color: #92400e; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #f59e0b; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        💡 Para Dummies • Módulo 01
      </span><br>
      <span style="color: #78350f; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #b45309; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Big%20Data/01%20-%20Procesamiento%20y%20Almacenamiento%20Distribuido/Para%20Dummies/01_MapReduce_Dummies.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## ¿Qué vamos a aprender aquí? 🎈

Este cuaderno es la versión **«para no ingenieros»** del [cuaderno 01 del Módulo 01 (MapReduce y sus limitaciones)](../01_MapReduce_y_sus_Limitaciones.ipynb). Imagina las **elecciones**: si **una sola persona** contara **5 000 papeletas** de todo el municipio, tardaría una eternidad. Lo que se hace es **repartir**: cada **mesa de votación** cuenta **solo sus papeletas** y entrega un **acta** con sus subtotales; luego las actas viajan a un punto central que **suma por candidato**. Eso es, casi palabra por palabra, **MapReduce**:

| En las elecciones | En MapReduce |
|---|---|
| Cada **mesa cuenta sus propias papeletas** | **Map**: cada trozo de los datos se procesa por separado y en paralelo |
| La mesa **resume en un acta** («Ana 380, Beto 120...») antes de enviarla | **Combiner**: reducir localmente para mandar menos |
| Las actas se **agrupan por candidato**: las de Ana a una ventanilla, las de Beto a otra | **Shuffle** (barajado): agrupar por clave |
| En cada ventanilla se **suman** las actas de un candidato | **Reduce**: combinar todos los valores de una clave |

Al terminar podrás explicar:
1. Por qué **contar por mesa y sumar actas** da **el mismo resultado** que contar todo junto, pero **repartiendo el trabajo**.
2. Por qué **una ventanilla saturada** (un candidato con muchísimos votos) **frena todo el conteo** y cómo se arregla.
3. Por qué **volver a escribir todo en papel en cada ronda** hace lento un trabajo que se repite muchas veces.

> 🔗 Si luego quieres ver el **motor MapReduce desde cero** con `multiprocessing`, el *join*, el índice invertido y los tiempos medidos (también contra Spark): [cuaderno estándar](../01_MapReduce_y_sus_Limitaciones.ipynb). Antes de esto: [dónde guardar los datos](00_Opciones_Procesamiento_Dummies.ipynb).

---
## 1. Cada Mesa Cuenta lo Suyo 🗳️

Tenemos **5 mesas** con **1 000 papeletas** cada una (datos **inventados**) para **4 candidatos**. Hacemos el conteo de dos maneras y comprobamos que **dan lo mismo**:

1. **Un solo jurado** cuenta las 5 000 papeletas.
2. **Cada mesa** cuenta las suyas (**Map**), escribe su **acta** con subtotales (**Combiner**) y el punto central **suma las actas** por candidato (**Reduce**).

Fíjate en **cuántas hojas** viajan al centro en cada caso: ¡**5 000 papeletas** frente a **20 renglones de actas**!

In [ ]:
import random
from collections import Counter

import matplotlib.pyplot as plt
import pandas as pd

random.seed(42)
candidatos = ["Ana", "Beto", "Carla", "Diego"]
# 5 000 papeletas inventadas (sin favoritismo extremo) repartidas en 5 mesas de 1 000
papeletas = random.choices(candidatos, weights=[35, 25, 25, 15], k=5000)
mesas = {f"Mesa {i + 1}": papeletas[i * 1000:(i + 1) * 1000] for i in range(5)}
assert all(len(v) == 1000 for v in mesas.values())

# 1) Un solo jurado
total_directo = Counter(papeletas)

# 2) Cada mesa cuenta lo suyo (map + combiner) y escribe su acta
actas = {nombre: Counter(votos) for nombre, votos in mesas.items()}
# Reduce: el punto central suma las actas por candidato
total_mapreduce = Counter()
for acta in actas.values():
    total_mapreduce.update(acta)

print(pd.DataFrame(actas).T.assign(**{"total mesa": lambda d: d.sum(axis=1)}).to_string())
print("\nTotal (suma de actas):", dict(total_mapreduce))
assert total_mapreduce == total_directo                                         # ¡el mismo resultado!
assert sum(total_mapreduce.values()) == 5000

hojas_sin_acta = len(papeletas)                                                 # cada papeleta viaja sola al centro
hojas_con_acta = sum(len(a) for a in actas.values())                            # un renglon por candidato por mesa
print(f"\nPapeletas que viajan al centro: SIN actas = {hojas_sin_acta:,} | CON actas = {hojas_con_acta} renglones ({hojas_sin_acta // hojas_con_acta} veces menos)")
assert hojas_con_acta == 20 and hojas_sin_acta // hojas_con_acta == 250

---
##### 🛠️ Práctica 1: Los votos de Carla, mesa por mesa

Con el diccionario `mesas` (nombre de la mesa → lista de papeletas) y `total_directo`:

1. Guarda en `subtotales` un diccionario `nombre de la mesa → votos de "Carla" en esa mesa` (pista: `votos.count("Carla")`). Esto es el **Map**.
2. Guarda en `total_carla` la **suma de los subtotales** de todas las mesas. Esto es el **Reduce**.
3. Comprueba con `assert` que `total_carla == total_directo["Carla"]` y que hay **5 subtotales**.

In [ ]:
# Escribe tu código aquí

# subtotales = {}
# total_carla = ...


<details>
<summary><b>💡 Haz clic aquí para ver la solución guiada...</b></summary>

```python
subtotales = {nombre: votos.count("Carla") for nombre, votos in mesas.items()}
total_carla = sum(subtotales.values())
print("Votos de Carla por mesa:", subtotales, "| total:", total_carla)

assert total_carla == total_directo["Carla"]
assert len(subtotales) == 5
```
</details>

---


---
**Lección:** cada mesa trabajó **por separado** (y **al mismo tiempo**, en la vida real) y la suma de las actas **coincide exactamente** con el conteo directo. Lo valioso es el **ahorro de viajes**: al centro llegaron **20 renglones en vez de 5 000 papeletas**. Esa es la razón de ser del *combiner*: **reducir antes de mover datos**, porque **mover es lo caro**.

---

## 2. La Ventanilla Saturada 🪟

Ahora imagina un candidato **muy popular**: **el 70 % de los votos** son de Ana. Las actas se agrupan **por candidato**: **todas** las papeletas de Ana van a **una sola ventanilla**, y esa ventanilla se llena mientras las otras tres casi no tienen trabajo. Como **el conteo termina cuando termina la ventanilla más lenta**, contratar más gente en las otras no ayuda.

Medimos la **carga** (papeletas que recibe cada ventanilla) en tres escenarios: **(a)** cada papeleta viaja sola; **(b)** las mesas mandan actas (el *combiner*); **(c)** dividimos la ventanilla de Ana en **5** ventanillas (una por mesa) y al final sumamos esos 5 subtotales (esto se llama **«salar» la clave**).

In [ ]:
# 5 000 papeletas: 70 % de Ana (3 500) y 500 para cada uno de los otros tres
papeletas2 = ["Ana"] * 3500 + ["Beto"] * 500 + ["Carla"] * 500 + ["Diego"] * 500
random.seed(7); random.shuffle(papeletas2)
mesas2 = {f"Mesa {i + 1}": papeletas2[i * 1000:(i + 1) * 1000] for i in range(5)}

def eficiencia(cargas):
    """Que tan bien se reparte: 1.0 = perfecto; baja cuanto mas se sature una ventanilla (promedio / maxima)."""
    return sum(cargas) / len(cargas) / max(cargas)

# (a) cada papeleta viaja sola: una ventanilla por candidato
carga_a = Counter(papeletas2)
# (b) las mesas mandan actas: cada ventanilla recibe 5 renglones (uno por mesa)
carga_b = {c: sum(1 for votos in mesas2.values() if c in votos) for c in candidatos}
# (c) ventanilla de Ana dividida en 5 (una por mesa), papeletas sueltas
grupo = {nombre: i for i, nombre in enumerate(mesas2)}
carga_c = Counter()
for nombre, votos in mesas2.items():
    for v in votos:
        carga_c[(v, grupo[nombre]) if v == "Ana" else (v, 0)] += 1
subtotal_ana = sum(n for (c, _), n in carga_c.items() if c == "Ana")             # el centro suma los 5 subtotales de Ana

print(f"(a) papeletas sueltas : {dict(carga_a)} | eficiencia {eficiencia(list(carga_a.values())):.0%} | aceleracion {5000 / max(carga_a.values()):.1f}x de 4x ideal")
print(f"(b) con actas         : {carga_b} | eficiencia {eficiencia(list(carga_b.values())):.0%}")
print(f"(c) Ana en 5 ventanillas: {dict(carga_c)} | eficiencia {eficiencia(list(carga_c.values())):.0%} | aceleracion {5000 / max(carga_c.values()):.1f}x con 8 ventanillas")

assert carga_a["Ana"] == 3500 and abs(eficiencia(list(carga_a.values())) - 1250 / 3500) < 1e-9
assert all(v == 5 for v in carga_b.values())                                    # con actas, cada ventanilla recibe solo 5 renglones
assert subtotal_ana == 3500 and max(carga_c.values()) < 800                    # ninguna ventanilla pasa de ~700 papeletas
assert eficiencia(list(carga_c.values())) > 2 * eficiencia(list(carga_a.values()))

---
**Lección:** con papeletas sueltas, **la ventanilla de Ana recibe 3 500** (el 70 %) y las demás solo 500: el conteo avanza apenas **1.4 veces más rápido** que con un solo jurado, **no 4**. Con **actas** (el *combiner*) el problema casi desaparece: cada ventanilla recibe **5 renglones**. Cuando **no se puede resumir** (por ejemplo, contar **personas distintas** que votaron), se **divide la ventanilla caliente** en varias y se **suma al final**: la eficiencia sube a más del doble. Moraleja: **el trabajo se reparte tan bien como se reparta la clave más popular.**

---

## 3. El Papel Cuesta: Cuando el Trabajo se Repite Muchas Veces 📝

Hay problemas que **se repiten por rondas** (por ejemplo, ir **ajustando un puntaje** una y otra vez hasta que se estabilice). Imagina que **cada ronda** de un conteo es **una elección completa** y que, al terminar cada una, **todo se escribe en actas de papel y se vuelve a leer** para la siguiente (así trabaja el MapReduce original: guarda en disco entre rondas). La alternativa es **dejar los resultados en una pizarra** (la memoria) para la ronda siguiente.

Un **modelo de juguete** con **supuestos inventados** (no son medidas reales): **contar** una ronda toma **2 minutos** y **escribir y volver a leer** las actas toma **3 minutos**.

In [ ]:
MIN_CONTAR = 2            # supuesto de juguete: minutos por ronda de conteo
MIN_PAPEL = 3             # supuesto de juguete: minutos por escribir y volver a leer las actas


def minutos_totales(rondas, usa_papel):
    return rondas * (MIN_CONTAR + (MIN_PAPEL if usa_papel else 0))


rondas = [1, 2, 5, 10, 20]
con_papel = [minutos_totales(r, True) for r in rondas]
con_pizarra = [minutos_totales(r, False) for r in rondas]
print(pd.DataFrame({"rondas": rondas, "con papel (min)": con_papel, "con pizarra (min)": con_pizarra}).to_string(index=False))
assert minutos_totales(1, True) == 5 and minutos_totales(10, True) == 50 and minutos_totales(10, False) == 20
assert all(p > m for p, m in zip(con_papel, con_pizarra))

fig, ax = plt.subplots(figsize=(7, 3.8))
ax.plot(rondas, con_papel, "o-", color="#dc2626", label="actas en papel entre rondas")
ax.plot(rondas, con_pizarra, "o-", color="#16a34a", label="resultados en la pizarra (memoria)")
ax.set_xlabel("número de rondas"); ax.set_ylabel("minutos"); ax.legend(); ax.set_title("El costo del papel crece con cada ronda (modelo de juguete)", fontweight="bold", fontsize=10)
plt.tight_layout(); plt.show()

---
**Lección:** con **una sola ronda** el papel casi no se nota (5 frente a 2 minutos); con **10 rondas** el trabajo en papel toma **50 minutos** frente a **20** con la pizarra, y la **diferencia crece con cada ronda**. Por eso **MapReduce es razonable para un solo paso** sobre muchísimos datos, pero **lento para algoritmos que repiten rondas**; ahí entra **Spark**, que deja los resultados **en la memoria** entre rondas. (Los minutos son **inventados** para ilustrar; en el cuaderno estándar se mide el costo real.)

---

## Resumen en una frase 🎯

> **MapReduce reparte un conteo entre muchas mesas (cada una cuenta lo suyo), manda actas en vez de papeletas y suma por candidato; funciona muy bien para un solo paso sobre muchos datos, pero se frena con una clave muy popular y se hace lento si repite muchas rondas escribiendo todo en papel.**

### Lo que aprendiste hoy:
- ✅ **Mesas + actas + suma central:** el resultado es **idéntico** al conteo directo (5 000 votos) con **20 renglones** viajando en vez de 5 000 papeletas.
- ✅ **La ventanilla saturada:** con el 70 % de los votos para Ana, **un candidato frena a todos**; resumir en actas o **dividir la ventanilla** lo arregla.
- ✅ **El papel entre rondas:** cada ronda en papel suma tiempo; con 10 rondas, 50 minutos frente a 20 (modelo de juguete).

> 🎓 **Siguiente paso:** [el cuaderno estándar](../01_MapReduce_y_sus_Limitaciones.ipynb) construye el motor MapReduce con `multiprocessing`, mide el sesgo, la escritura a disco, PageRank y los archivos pequeños. Luego, [la biblioteca con copias en varias sedes (HDFS, Spark y HBase)](02_Frameworks_Distribuidos_Dummies.ipynb).

---
### 🧠 ¿Ya lo tienes claro?

1. Con tus palabras: ¿por qué es mejor que cada mesa **mande un acta** y no todas sus papeletas? ¿Qué cosa **no** se podría resumir en un acta (pista: «el promedio de votos por persona» o «cuántas personas distintas votaron»)?
2. Si dos candidatos tuvieran el 45 % de los votos cada uno, ¿cuántas ventanillas útiles habría en la práctica? ¿Qué pasaría si **añades más ventanillas**?
3. Piensa en un trabajo de tu vida que **se repite muchas veces** (por ejemplo, recalcular un presupuesto cada semana). ¿Qué parte sería el «papel» y cuál la «pizarra»?

---
<div align="center">
  <p style="font-size: 0.9em; color: #78350f;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Big Data (Edición Didáctica Para Dummies)</i>
  </p>
</div>